<a href="https://colab.research.google.com/github/antagomir/scripts/blob/master/gLVIntro.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Tutorial introducing the generalized Lotka-Volterra model

## Introduction

In brief, the generalized Lotka-Volterra (gLV) describes how species abundances change as a function of species' growth rates and their (pairwise) interactions. As a refresher, here is the model, where $x_i$ represents the abundance of species $i$, $r_i$ is the growth rate of species $i$, $n$ is the species number and $\alpha_{ij}$ represents the impact of species $j$ on species $i$.

$$\frac{dx_i}{dt} = x_i \left( r_i + \sum_{j=1}^{n} \alpha_{ij} x_j \right)$$

This tutorial will show how to run simulations with the gLV model using the [miaSim](https://bioconductor.org/packages/release/bioc/html/miaSim.html) R package.
We start by installing and loading miaSim from Bioconductor and the [vegan](https://cran.r-project.org/web/packages/vegan/index.html) R package from CRAN. This will take a bit of time (count on ca. 10 minutes), since all the dependencies have to be installed as well.

In [ ]:
if (!require("BiocManager", quietly = TRUE)) install.packages("BiocManager")
BiocManager::install("miaSim")
library(miaSim)
install.packages("vegan")
library(vegan)

## First gLV simulation

To run our first simulation, we need to define parameters and initial conditions. The gLV model takes two parameters: the vector of growth rates and the matrix of interaction strengths. We will generate them randomly for 10 species.

In [ ]:
N=10 # we want 10 species
d=-0.5 # strength of intra-specific competition
c=0.2 # percentage of non-zero off-diagonal entries in the interaction matrix

# generate the parameters
A=randomA(n_species=N, diagonal=d, connectance = c) # interaction matrix
y=bstick(N,5) # generate uneven growth rates with a stick-breaking process implemented in vegan

# assign species names to growth rate vector
names(y)=rownames(A)

# plot the parameters

# we define a helper function to plot A
# colors range from orange (negative) via white (0) to blue (positive)
plotA<-function(A,lim=max(abs(A)), title=""){
  breaks=seq(-lim, lim, length.out = 101)
  negCols=colorRampPalette(c("orange", "white"))(50)
  posCols=colorRampPalette(c("white", "blue"))(50)
  image(1:ncol(A), 1:nrow(A), t(A[nrow(A):1, ]), col = c(negCols, posCols), breaks = breaks, axes = FALSE, xlab = "", ylab = "", main=title)
  axis(1, at = 1:ncol(A), labels = colnames(A), las = 2)
  axis(2, at = 1:nrow(A), labels = rev(rownames(A)), las = 2)
}
plotA(A)
barplot(y,ylab = "Growth rates")


Next, we generate the initial abundances. Here, we'll give each species the same initial abundance. Then we are ready to launch a simulation.

In [ ]:
# total community count evenly distributed across species
totalcount=10
x=rep(round(totalcount/N),N) # even initial abundances

# run a simulation
T=500
# we provide initial abundances x, species number, interaction matrix, growth rates and end time point
# we also switch off miaSim-specific gLV extensions (stochasticity and migration)
glv.sim = simulateGLV(x0=x, n_species = N, A = A, growth_rates=y, t_end = T, stochastic = FALSE, migration_p=0)

# plot the result
counts=t(glv.sim@assays@data$counts)
matplot(counts[1:T,],type="l",xlab="Time",ylab="Species abundance")
legend("topright", colnames(counts), col=1:N, cex=0.8, lty=1:N, lwd=2)

Do you see what drives the steady state here?

In this simulation, we assigned the same negative value to all the diagonal entries of the interaction matrix, which represent intra-species competition. <br>
<font color="red">Task 1</font>: What happens if you introduce a positive or zero value for a diagonal entry? Try it below.

In [ ]:
# your code for Task 1 goes here

As you saw when completing Task 1, intra-species competition is required to prevent explosions. However, if they are set too strongly, they dominate the dynamics, so selecting sensible values for the diagonal is a critical step in gLV simulations.

## GLV simulation with noise

Next, let's make the simulation a bit more realistic by working with unequal initial abundances and adding stochasticity in the form of small noise. The noise is added during the simulation and can be interpreted as small environmental impact on the species.

In [ ]:
x=sample(bstick(N,totalcount)) # distribute total community count unevenly among species and shuffle
names(x)=rownames(A)
glv.sim = simulateGLV(x0=x, n_species = N, A = A, growth_rates=y, t_end = T, stochastic = TRUE, migration_p=0)

# plot the result
par(mfrow=c(1,2))
barplot(x,ylab="Initial abundances")
barplot(y,ylab="Growth rates")
par(mfrow=c(1,1))

counts=t(glv.sim@assays@data$counts)
matplot(counts[1:T,],type="l",xlab="Time",ylab="Species abundance")
legend("topright", colnames(counts), col=1:N, cex=0.8, lty=1:N, lwd=2)



We see that the dynamics is still dominated by the growth rates. What can we do about it?

We can equalize growth rates, but that is not very realistic. We can also strengthen the interactions by increasing their absolute values. Here, we will try the latter.

In [ ]:
# we increase both the number of interactions (with connectance) and their strength (with scale_off_diagonal, which defaults to 0.1)
A=randomA(n_species=10, diagonal=d, connectance = 0.5, scale_off_diagonal = 0.8) # interaction matrix

glv.sim = simulateGLV(x0=x, n_species = N, A = A, growth_rates=y, t_end = T, stochastic = TRUE, migration_p=0)

# plot the result
par(mfrow=c(1,2))
barplot(x,ylab="Initial abundances")
barplot(y,ylab="Growth rates")
par(mfrow=c(1,1))

plotA(A)

counts=t(glv.sim@assays@data$counts)
matplot(counts[1:T,],type="l",xlab="Time",ylab="Species abundance")
legend("topright", colnames(counts), col=1:N, cex=0.8, lty=1:N, lwd=2)


## Example application: community types

Now we are ready for an example application. A widespread use of the gLV model is to generate synthetic data to test hypotheses. We will illustrate this using [the work by Gibson and colleagues](https://journals.plos.org/ploscompbiol/article?id=10.1371/journal.pcbi.1004688) on the origin of community types. When analysing microbial compositions of fecal samples, [Arumugam and colleagues](https://www.nature.com/articles/nature09944) noted that they form different clusters (enterotypes). Below is a figure from [Falony and colleagues](https://www.science.org/doi/10.1126/science.aad3503), which shows enterotypes emerging in an ordination combined with a density plot. One dot in the ordination represents one community composition observed in a sample. The color encodes the community type (i.e. whether the sample is dominated by Ruminococcacaea, Bacteroidaceae or Prevotellacaea) and the third dimension visualizes how many samples are present in each region of the plot.
<img src="https://drive.google.com/uc?export=view&id=1vdc9nMq4_gc4q-YBimVHeCnvFUUGaMtF" width="400"><br>
The clusters are not perfectly separated, but they are a phenomenon in need of an explanation. Gibson and colleagues proposed that strongly interacting species (SIS) could explain the existence of community types. They assume that local gut communities in different individuals are randomly assembled from a global species pool, the meta-community.<br>
<img src="https://drive.google.com/uc?export=view&id=16esB-2TM_jBAYhRlKRCEQJ44qAYRI4k6" width="400">
<br>
Local communities that include SIS are expected to cluster apart from communities that do not, since SIS affect community composition through their strong interactions with other community members. Gibson and colleagues tested this hypothesis with simulations using the gLV model. We will reproduce their work in simplified form.

Here, we assume that our interaction matrix A covers the interactions between all species in the global species pool. We then need a way to sub-sample local communities from the global pool.<br>
<font color="red">Task 2</font>: You can think about how to implement this. Tip: you can introduce a constant M that encodes the host number (so the number of local communities) and use the sample() function to obtain the indices of random species sub-sets.

In [ ]:
# your code for Task 2 goes here

Once we are able to sub-sample randomly from the global species pool, we can run a simulation for each species subset, as shown below. We can keep the last time point from each simulation to represent the resulting local community composition, assuming that a steady state was reached.

In [ ]:
M=15
subset=N/2
localCompositions=matrix(nrow=N,ncol=M,0) # initialize matrix with zeros
# loop over the number of local communities we want to generate
for(i in 1 : M){
  localSpecies=sample(1:N)[1:subset]
  localA=A[localSpecies,localSpecies]
  local.glv=simulateGLV(x0=x[localSpecies], n_species = subset, A = localA, growth_rates=y[localSpecies], t_end = T, stochastic = TRUE, migration_p=0)
  counts=t(local.glv@assays@data$counts)
  localCompositions[localSpecies,i]=counts[T,]
}

We can now use the capscale function in vegan to carry out an ordination, namely a principal coordinate analysis (PCoA), on the local community compositions. Ordination reduces community composition to a few dimensions, which makes it easier to check visually whether clusters emerge. To see the link between interaction strengths and clusters, we also plot the interaction strength distribution in the global interaction matrix using absolute values.  

In [ ]:
par(mfrow=c(1,2))
hist(abs(A), main="Interaction strengths")
pcoa.res=capscale(data.frame(t(localCompositions))~1,distance="bray", na.action = "na.omit", na.rm=TRUE)
plot(pcoa.res$CA$u[,c(1,2)])
par(mfrow=c(1,1))

The hypothesis by Gibson and colleagues is that a highly uneven distribution of interaction strengths in the global interaction matrix introduced by SIS leads to clusters. <br>
<font color="red">Task 3</font>: Test this assumption. You can use the powerA function defined below to generate As with different interaction strength distributions. Alpha set to 7 will give an interaction matrix with more evenly distributed interaction strengths and alpha set to 1.01 will give a highly uneven interaction strengh distribution. Bonus: you can identify SIS in the strongly uneven interaction matrix and color samples in the PCoA plot separately according to their presence or absence.

In [ ]:
# Implements the Gibson et al. approach to generate interaction matrices with uneven interaction strength distributions
powerA<-function(n_species,alpha){
  u <- runif(n_species, min = 0, max = 1 - .Machine$double.eps^0.5) # generate random values between zero and one
  power <- (1 - u)^(1 / (1 - alpha)) # generate weights according to a power law; the closer alpha is to 1, the larger the values
  H <- power/mean(power)  # weights are normalized so mean is 1
  N <- matrix(rnorm(n = n_species^2, mean = 0, sd = 1), nrow = n_species) # interaction strengths are sampled from a normal distribution
  diag(N) <- 0
  interactions_custom <- N %*% diag(H) # matrix multiplication of random interaction strengths with weights, will generate SIS for low alpha
  A <- randomA(
    n_species = n_species,
    diagonal = -1,
    scale_off_diagonal = 0.07,
    connectance = 1, # all non-diagonal entries are non-zero
    interactions = interactions_custom
  )
  return(A)
}

N=100 # work with a large global community of 100 species
subset=80 # number of species in local commmunities
M=50 # number of local communities
T=100 # simulation end
y=runif(n = N, min = 0, max = 1) # random global growth rates
x=runif(n = N, min = 0, max = 1) # random global initial abundances
# your code for Task 3 goes here....


## Example application: evaluation of network inference

Another typical application of the gLV model is the generation of synthetic abundances with known parameters to test network inference algorithms. To illustrate this, we will generate cross-sectional abundance data with a small species and sample number, to keep run time reasonable.<br>
<font color="red">Task 4</font>: Go ahead and generate an abundance matrix for 50 species and 30 samples using the same interaction matrix for each sample. To generate varying community compositions, you can sub-sample from a global species pool or just rely on variable initial abundances and growth rates. You can also play with randomA parameters to change the proportion of different interaction types.

In [ ]:
# your code for Task 4 goes here....


We would normally have to filter and sum species that are not sufficiently prevalent but we skip this step here. Sequencing data also have to be normalized in some form to take out the effect of varying sequencing depth. We do this here by converting absolute to relative abundances.  We then carry out a rudimentary network inference with Pearson correlation. There are dozens of sophisticated network inference algorithms that could be used instead. You can find an introduction to microbial network inference [here](https://www.cell.com/trends/microbiology/fulltext/S0966-842X(25)00027-7) and [here](https://www.nature.com/articles/s41396-021-01027-4).

In [ ]:
# comData: the abundance matrix generated in Task 4
comDataNorm=sweep(comData,2,STATS=colSums(comData),FUN="/") # convert absolute into relative abundances
network=cor(t(comDataNorm)) # basic network inference: compute all pair-wise Pearson correlations between species
filter=max(abs(network))/4 # filter out edges with low correlation (here an arbitrary threshold)
network[abs(network)<filter]=0 # rough filter: set edges with low correlations to zero

We now visualise the true interaction matrix next to the inferred one.

In [ ]:
par(mfrow=c(1,2))
lim=max(c(max(abs(A)),max(abs(network)))) # make sure the color scale spans both networks
plotA(A,title="True",lim=lim) # A is the network you used in Task 4
plotA(network,title="Inferred",lim=lim)
par(mfrow=c(1,1))

# we convert the matrix entries into 3 values (-1, 0 and 1) to make the comparison easier
A2=A
network2=network
A2[A2>0]=1
A2[A2<0]=-1
network2[network2>0]=1
network2[network2<0]=-1
plotA(A2,title="True")
plotA(network2,title="Inferred")


How accurate is the inference? Which issues do you see?

One obvious problem is that the correlation matrix is symmetric, while the true interaction matrix is not. You can think about ways to compare the two matrices, but it is clear by eye that the inferred network does not have much to do with the true interaction network. You can use what you have learned here to test your favourite network inference algorithm. If you do so, please also add noise after the inference to represent measurement errors and keep in mind that the gLV model is not the only model you can use to generate microbial abundance data. [miaSim](https://microbiome.github.io/miaSim/) also implements a couple of other models that you can take a look at, such as the consumer-resource model and the neutral model.
<br>
That was it. Have fun implementing your own application of the gLV model 😀.